# L2 trace optimization on simulated training subjects

Optimize the mean per-spectrum relative complex L2 error against **metabolites + the same receiver noise**:
`||input - input @ P - water - (metabolites + noise)|| / ||metabolites + noise||`.
Equivalently, the numerator is `||lipids - input @ P||`.
Only training subjects are used, with equal numbers of spectra and equal subject weights.
All trace candidates use identical simulations. No network or checkpoint is needed.

The operator uses the original mask's stored water-suppressed lipid FIDs, as in
`EvaluateYNetL2Input`. One complex128 eigendecomposition per subject is reused.
Norms are evaluated in the orthonormal eigenbasis (unitary invariance), avoiding
one dense matrix multiplication per trace and spectrum.


In [ ]:
from pathlib import Path
from dataclasses import replace
import sys
import time
import json
import h5py
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from walinet.config.build import build_config
from walinet.config.build_simulation import build_simulation_config
from walinet.training_data.simulation_resources import load_simulation_pool
from walinet.training_data.lcmodel_basis.acquisition import prepare_basis_for_acquisition
from walinet.training_data.metabolite_simulation import MetaboliteSimulator
from walinet.training_data.spectrum_simulator import SpectrumSimulator

# ---- Change these settings ------------------------------------------------
TRAINING_CONFIG = PROJECT_ROOT / "configs/Training/train_7T.yaml"
TRACE_MIN = 0.9
TRACE_MAX = 1.0
N_TRACE_POINTS = 101                 # inclusive endpoints; default step 0.001
N_SPECTRA_PER_SUBJECT = 10000
BATCH_SIZE = 500                     # simulation and evaluation memory limit
SEED = 24680
GPU_INDEX = 0
TRACE_TOL = 1e-6
PROGRESS_EVERY = 10                  # completed trace points per subject
SUBJECTS = None                      # None = all train_subjects; or a subset
SAVE_GT = True                       # complex input, water, lipids, metabolites, noise
OUTPUT_PATH = PROJECT_ROOT / "results/l2_trace_7T.h5"  # existing files are protected

# Initially evaluate one acquisition length. None uses the simulation maximum.
# For future 3T runs, set the desired native length explicitly (e.g. 288).
# No random acquisition-length mixing: each scan needs a matching operator.
ACQUISITION_LENGTH = None


In [ ]:
def read_yaml(path):
    return yaml.safe_load(Path(path).read_text())

train_cfg = build_config(read_yaml(TRAINING_CONFIG), config_dir=TRAINING_CONFIG.parent)
sim_path = Path(train_cfg.data.simulation_config)
sim_cfg = build_simulation_config(read_yaml(sim_path), config_dir=sim_path.parent)
# The sweep itself computes the second input; do not load/use cached operators.
sim_cfg = replace(sim_cfg, lipid_projection=replace(sim_cfg.lipid_projection, enabled=False))
subjects = list(train_cfg.data.train_subjects if SUBJECTS is None else SUBJECTS)
if not subjects or len(set(subjects)) != len(subjects) or not set(subjects) <= set(train_cfg.data.train_subjects):
    raise ValueError("SUBJECTS must be a nonempty unique subset of train_subjects")
if not (0 < TRACE_MIN < TRACE_MAX <= 1 and N_TRACE_POINTS >= 2):
    raise ValueError("Require 0 < TRACE_MIN < TRACE_MAX <= 1 and at least two points")
if min(N_SPECTRA_PER_SUBJECT, BATCH_SIZE, PROGRESS_EVERY) < 1:
    raise ValueError("Counts and progress interval must be positive")
traces = np.linspace(TRACE_MIN, TRACE_MAX, N_TRACE_POINTS)
if not 0 < TRACE_TOL < (TRACE_MAX - TRACE_MIN) / (N_TRACE_POINTS - 1) / 10:
    raise ValueError("TRACE_TOL must be positive and much smaller than the grid spacing")
n_native = ACQUISITION_LENGTH or sim_cfg.acquisition.n_timepoints
if not sim_cfg.acquisition.min_acquired_n_timepoints <= n_native <= sim_cfg.acquisition.max_acquired_n_timepoints:
    raise ValueError("ACQUISITION_LENGTH must lie within the simulation config range")
n_points = sim_cfg.acquisition.n_timepoints if sim_cfg.acquisition.zero_filling else n_native
device = torch.device(f"cuda:{GPU_INDEX}" if torch.cuda.is_available() else "cpu")
resource_filename = train_cfg.data.resources.filename.format(version=train_cfg.data.resources.version)
print(f"Device: {device}; {len(subjects)} subjects; {N_SPECTRA_PER_SUBJECT} spectra/subject")
print(f"{len(traces)} trace points: {traces[0]:.6f} to {traces[-1]:.6f}")
print(f"Fixed acquisition: {n_native} native samples, {n_points} output points")
print(f"Output: {OUTPUT_PATH}")


In [ ]:
def acquire(spectra):
    """Apply the same fixed acquisition length as the simulator to GT components."""
    if n_native == spectra.shape[-1]:
        return spectra
    fid = torch.fft.ifft(torch.fft.ifftshift(spectra, dim=-1), dim=-1)
    if sim_cfg.acquisition.zero_filling:
        fid = fid.clone()
        fid[..., n_native:] = 0
    else:
        fid = fid[..., :n_native]
    return torch.fft.fftshift(torch.fft.fft(fid, dim=-1), dim=-1)


def operator_eigensystem(subject):
    subject_dir = Path(train_cfg.data.base_dir) / subject
    with h5py.File(subject_dir / resource_filename, "r") as h5:
        fids = np.asarray(h5["lipid_fids"], dtype=np.complex128)
    mask = np.load(subject_dir / "masks/lipid_mask.npy") > 0
    if len(fids) != np.count_nonzero(mask):
        raise ValueError(f"{subject}: stored lipid FIDs do not match the original mask")
    if not len(fids) or not np.isfinite(fids).all():
        raise ValueError(f"{subject}: invalid lipid FIDs")
    # Match the viewer: truncate or zero-pad stored water-suppressed FIDs.
    prepared = np.zeros((len(fids), n_points), dtype=np.complex128)
    count = min(fids.shape[-1], n_points)
    prepared[:, :count] = fids[:, :count]
    a = torch.as_tensor(np.fft.fftshift(np.fft.fft(prepared, axis=-1), axes=-1), device=device)
    eigenvalues, eigenvectors = torch.linalg.eigh(a.mH @ a)
    scale = eigenvalues.abs().max()
    if scale <= 0:
        raise ValueError(f"{subject}: zero lipid matrix")
    # Treat eigenvalues at roundoff scale as null-space directions.
    cutoff = torch.finfo(torch.float64).eps * n_points * scale
    if eigenvalues.min() < -10 * cutoff:
        raise RuntimeError("Lipid Gram matrix has significant negative eigenvalues")
    eigenvalues = torch.where(eigenvalues > cutoff, eigenvalues, 0)
    return eigenvalues, eigenvectors


def projection_weights(eigenvalues, target):
    """Find beta using normalized eigenvalues; trace=1 is exactly P=0."""
    scale = eigenvalues.max()
    lam = eigenvalues / scale
    floor = float((lam == 0).double().mean())
    if target == 1:
        return torch.zeros_like(lam), 0.0, 1.0
    if target <= floor:
        raise ValueError(f"Trace {target:g} is unreachable; null-space floor={floor:g}")
    def trace_at(b):
        return float((1 / (1 + b * lam)).mean())
    lo, hi = 0.0, 1.0
    for _ in range(1024):
        if trace_at(hi) <= target:
            break
        hi *= 2
    else:
        raise RuntimeError("Could not bracket beta")
    for _ in range(200):
        b = (lo + hi) / 2
        actual = trace_at(b)
        if abs(actual - target) <= TRACE_TOL:
            return b * lam / (1 + b * lam), b / float(scale), actual
        if actual > target:
            lo = b
        else:
            hi = b
    raise RuntimeError("Trace bisection did not converge")


In [ ]:
# Run the sweep. This cell writes incrementally; a failed run has complete=False.
# Choose another OUTPUT_PATH before rerunning to preserve previous results.
if OUTPUT_PATH.exists():
    raise FileExistsError(f"Choose a new OUTPUT_PATH: {OUTPUT_PATH}")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
basis = prepare_basis_for_acquisition(
    sim_cfg.basis.library, target_bandwidth=sim_cfg.acquisition.bandwidth_hz,
    target_n_timepoints=sim_cfg.acquisition.n_timepoints, dataset_name="clean_fid")
metabolite_simulator = MetaboliteSimulator(prepared_basis=basis, config=sim_cfg, device=device)
subject_errors = np.empty((len(subjects), len(traces)))
started = time.monotonic()
with torch.no_grad(), h5py.File(OUTPUT_PATH, "x") as out:
    out.attrs["complete"] = False
    out.attrs["metric"] = "mean(||lipids-input@P||_2 / ||metabolites+noise||_2)"
    out.attrs["seed"] = SEED
    out.attrs["n_spectra_per_subject"] = N_SPECTRA_PER_SUBJECT
    out.attrs["batch_size"] = BATCH_SIZE
    out.attrs["trace_tolerance"] = TRACE_TOL
    out.attrs["native_length"] = n_native
    out.attrs["training_config"] = str(TRAINING_CONFIG)
    out.create_dataset("training_config_yaml", data=TRAINING_CONFIG.read_text())
    out.create_dataset("simulation_config_yaml", data=sim_path.read_text())
    out.create_dataset("trace_targets", data=traces)
    groups = out.create_group("subjects")
    for si, subject in enumerate(subjects):
        print(f"[{si+1}/{len(subjects)}] {subject}: computing eigendecomposition", flush=True)
        eigenvalues, q = operator_eigensystem(subject)
        candidates = [projection_weights(eigenvalues, float(t)) for t in traces]
        group = groups.create_group(subject)
        group.create_dataset("beta", data=[c[1] for c in candidates])
        group.create_dataset("achieved_trace", data=[c[2] for c in candidates])
        pool = load_simulation_pool(
            base_dir=train_cfg.data.base_dir, subjects=[subject], resource_filename=resource_filename,
            target_n_timepoints=sim_cfg.acquisition.n_timepoints,
            expected_bandwidth_hz=sim_cfg.acquisition.bandwidth_hz,
            load_projection_operator=False).to(device)
        simulator = SpectrumSimulator(pool=pool, metabolite_simulator=metabolite_simulator, config=sim_cfg)
        generator = torch.Generator(device=device).manual_seed(SEED + list(train_cfg.data.train_subjects).index(subject))
        # CPU eigenbasis cache: GPU memory remains bounded by BATCH_SIZE.
        xq = np.empty((N_SPECTRA_PER_SUBJECT, n_points), dtype=np.complex128)
        lq = np.empty_like(xq)
        denominator = np.empty(N_SPECTRA_PER_SUBJECT, dtype=np.float64)
        if SAVE_GT:
            gt = group.create_group("spectra")
            for name in ("input", "water", "lipids", "metabolites", "noise"):
                gt.create_dataset(name, shape=(N_SPECTRA_PER_SUBJECT, n_points), dtype=np.complex64,
                                  chunks=(min(BATCH_SIZE, N_SPECTRA_PER_SUBJECT), n_points))
        for start in range(0, N_SPECTRA_PER_SUBJECT, BATCH_SIZE):
            stop = min(start + BATCH_SIZE, N_SPECTRA_PER_SUBJECT)
            raw = simulator.simulate_raw(batch_size=stop-start, generator=generator,
                                         acquisition_length_override=n_native)
            x = raw.input_spectra
            w = acquire(raw.assembled.water_spectra)
            l = acquire(raw.assembled.lipid_spectra)
            m = acquire(raw.assembled.clean_metabolite_spectra)
            # Residual recovers the actual scaled receiver noise after acquisition.
            noise = x - acquire(raw.assembled.clean_mixture_spectra)
            reference = (m + noise).to(torch.complex128)
            den = torch.linalg.vector_norm(reference, dim=-1)
            if not torch.isfinite(den).all() or torch.any(den <= 0):
                raise ValueError("Invalid GT reference norm")
            xq[start:stop] = (x.to(torch.complex128) @ q).cpu().numpy()
            lq[start:stop] = (l.to(torch.complex128) @ q).cpu().numpy()
            denominator[start:stop] = den.cpu().numpy()
            if SAVE_GT:
                for name, value in dict(input=x, water=w, lipids=l, metabolites=m, noise=noise).items():
                    gt[name][start:stop] = value.cpu().numpy()
            print(f"  Simulation: {stop}/{N_SPECTRA_PER_SUBJECT} spectra", flush=True)
        errors = group.create_dataset("relative_l2", shape=(len(traces), N_SPECTRA_PER_SUBJECT), dtype=np.float64)
        for ti, (weights, beta, actual) in enumerate(candidates):
            for start in range(0, N_SPECTRA_PER_SUBJECT, BATCH_SIZE):
                stop = min(start+BATCH_SIZE, N_SPECTRA_PER_SUBJECT)
                xx = torch.as_tensor(xq[start:stop], device=device)
                ll = torch.as_tensor(lq[start:stop], device=device)
                values = torch.linalg.vector_norm(ll - xx * weights, dim=-1).cpu().numpy() / denominator[start:stop]
                if not np.isfinite(values).all():
                    raise ValueError("Non-finite relative error")
                errors[ti, start:stop] = values
            subject_errors[si, ti] = np.mean(errors[ti])
            if (ti+1) % PROGRESS_EVERY == 0 or ti+1 == len(traces):
                print(f"  {subject}: {ti+1}/{len(traces)} trace points calculated "
                      f"(all subjects: {si*len(traces)+ti+1}/{len(subjects)*len(traces)}; "
                      f"elapsed {time.monotonic()-started:.0f}s)", flush=True)
                out.flush()
        group.create_dataset("mean_relative_l2", data=subject_errors[si])
        del pool, simulator, xq, lq, q, candidates
    mean_error = subject_errors.mean(axis=0)
    out.create_dataset("mean_relative_l2", data=mean_error)
    out.attrs["best_trace"] = traces[np.argmin(mean_error)]
    out.attrs["complete"] = True
print(f"Finished in {time.monotonic()-started:.1f}s; saved {OUTPUT_PATH}")


In [ ]:
# Small numerical check before starting the expensive simulation.
with torch.no_grad():
    test_generator = torch.Generator(device=device).manual_seed(123)
    a = torch.randn(16, 8, dtype=torch.complex128, device=device, generator=test_generator)
    x = torch.randn(5, 8, dtype=torch.complex128, device=device, generator=test_generator)
    l = torch.randn(5, 8, dtype=torch.complex128, device=device, generator=test_generator)
    gram = a.mH @ a
    ev, q_test = torch.linalg.eigh(gram)
    identity = torch.eye(8, dtype=torch.complex128, device=device)
    for target in (0.9, 0.938, 0.99, 1.0):
        weights, beta, achieved = projection_weights(ev, target)
        p_direct = identity - torch.linalg.inv(identity + beta * gram)
        direct = torch.linalg.vector_norm(l - x @ p_direct, dim=-1)
        fast = torch.linalg.vector_norm(l @ q_test - (x @ q_test) * weights, dim=-1)
        torch.testing.assert_close(direct, fast, rtol=1e-10, atol=1e-10)
        assert abs(achieved - target) <= TRACE_TOL
print("Numerical self-check passed: eigenbasis and direct operator agree.")


In [ ]:
# Can also be run independently after loading the imports and settings cells.
with h5py.File(OUTPUT_PATH, "r") as out:
    if not out.attrs["complete"]:
        raise RuntimeError("This sweep is incomplete; inspect the progress above")
    targets = out["trace_targets"][:]
    overall = out["mean_relative_l2"][:]
    fig, ax = plt.subplots(figsize=(10, 5))
    for subject, group in out["subjects"].items():
        ax.plot(targets, group["mean_relative_l2"][:], alpha=0.4, lw=1, label=subject)
    best = int(np.argmin(overall))
    ax.plot(targets, overall, color="black", lw=2.5, label="Subject-balanced mean")
    ax.axvline(targets[best], color="black", linestyle="--", label=f"Best: {targets[best]:.6f}")
    ax.set(xlabel="Target normalized trace of residual operator", ylabel="Mean relative L2 error")
    ax.grid(alpha=0.25)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    plt.show()
    print(f"Best trace: {targets[best]:.6f}; mean relative error: {overall[best]:.6g}")
    if best == 0:
        print("Minimum at lower search boundary: consider extending the range downward.")
    elif best == len(targets)-1:
        print("Minimum at upper search boundary; trace=1 means no lipid removal.")
    else:
        print(f"Optional refinement interval: [{targets[best-1]:.6f}, {targets[best+1]:.6f}]")
